In [0]:
%pip install torch scikit-learn --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.pytorch
import torch
import numpy as np
import pandas as pd
from pyspark.sql.functions import col
from sklearn.preprocessing import StandardScaler

SILVER_TABLE = "aml_pipeline.transactions.silver_transactions"
GOLD_TABLE   = "aml_pipeline.transactions.gold_sar_reports"
MODEL_URI    = "models:/workspace.default.sentinelflow_fraudgnn/1"

# Step 1: Load pre-trained model
print("Loading GNN model from MLflow registry...")
model = mlflow.pytorch.load_model(MODEL_URI)
model.eval()
print("Model loaded successfully")

# Step 2: Load Silver table
print("\nLoading Silver table...")
silver_pd = spark.table(SILVER_TABLE).select(
    "transaction_id", "amount_usd", "batch_number",
    "high_risk_country", "large_transaction", "is_flagged"
).toPandas()
print(f"Loaded {len(silver_pd):,} transactions")

# Step 3: Build feature matrix (165 features)
n = len(silver_pd)
f1 = (silver_pd["amount_usd"] / silver_pd["amount_usd"].max()).fillna(0).values
f2 = silver_pd["high_risk_country"].astype(float).fillna(0).values
f3 = silver_pd["large_transaction"].astype(float).fillna(0).values
f4 = silver_pd["is_flagged"].astype(float).fillna(0).values
f5 = (silver_pd["batch_number"] / 11.0).fillna(0).values
np.random.seed(42)
noise   = np.random.randn(n, 160) * 0.1
X_score = np.column_stack([f1, f2, f3, f4, f5, noise]).astype(np.float32)
X_score = StandardScaler().fit_transform(X_score)

# Step 4: Score in batches of 10,000
print("\nScoring transactions...")
all_scores = []
for i in range(0, len(X_score), 10_000):
    batch = torch.tensor(X_score[i:i+10_000], dtype=torch.float32)
    with torch.no_grad():
        scores = model(batch).numpy()
    all_scores.extend(scores.tolist())
    print(f"  Scored {min(i+10_000, len(X_score)):,} / {len(X_score):,}")

silver_pd["fraud_risk_score"] = [round(float(s), 4) for s in all_scores]

# Step 5: Write scores back to Silver and Gold
print("\nWriting fraud scores to Silver table...")
scores_spark = spark.createDataFrame(silver_pd[["transaction_id", "fraud_risk_score"]])

(spark.table(SILVER_TABLE)
     .drop("fraud_risk_score")
     .join(scores_spark, on="transaction_id", how="left")
     .write.format("delta").mode("overwrite")
     .option("mergeSchema", "true")
     .saveAsTable(SILVER_TABLE))

print("Updating Gold SAR reports...")
(spark.table(GOLD_TABLE)
     .drop("fraud_risk_score")
     .join(scores_spark, on="transaction_id", how="left")
     .write.format("delta").mode("overwrite")
     .option("mergeSchema", "true")
     .saveAsTable(GOLD_TABLE))

# Step 6: Verify
print("\nVerification:")
spark.table(SILVER_TABLE).selectExpr(
    "count(*) as total",
    "count(fraud_risk_score) as scored",
    "round(avg(fraud_risk_score),4) as avg_score",
    "round(max(fraud_risk_score),4) as max_score",
    "count(case when fraud_risk_score > 0.7 then 1 end) as high_risk"
).show()

print("Top 5 highest fraud risk SARs:")
spark.table(GOLD_TABLE) \
    .orderBy("fraud_risk_score", ascending=False) \
    .select("sar_reference", "sender_name", "amount_usd", "flag_reason", "fraud_risk_score") \
    .show(5, truncate=False)

print("GNN scoring complete!")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a5fbb4a8-b399-4907-a7ce-6bdced79c63d/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


Loading GNN model from MLflow registry...


Model loaded successfully

Loading Silver table...
Loaded 100,000 transactions

Scoring transactions...
  Scored 10,000 / 100,000
  Scored 20,000 / 100,000
  Scored 30,000 / 100,000
  Scored 40,000 / 100,000
  Scored 50,000 / 100,000
  Scored 60,000 / 100,000
  Scored 70,000 / 100,000
  Scored 80,000 / 100,000
  Scored 90,000 / 100,000
  Scored 100,000 / 100,000

Writing fraud scores to Silver table...
Updating Gold SAR reports...

Verification:
+------+------+---------+---------+---------+
| total|scored|avg_score|max_score|high_risk|
+------+------+---------+---------+---------+
|100000|100000|   0.1855|      1.0|    16611|
+------+------+---------+---------+---------+

Top 5 highest fraud risk SARs:
+---------------------+----------------+----------+---------------------------------------------+----------------+
|sar_reference        |sender_name     |amount_usd|flag_reason                                  |fraud_risk_score|
+---------------------+----------------+----------+-------